In [ ]:
import requests
from datetime import datetime, timezone
from pyspark.sql import functions as F

CONTROL_TABLE = "ctl_monthly_load"
BASE_URL = "https://transtats.bts.gov/PREZIP/"
FILE_PREFIX = "On_Time_Reporting_Carrier_On_Time_Performance_1987_present_"
MAX_MONTHS_TO_CHECK = 6

control = spark.table(CONTROL_TABLE)
existing = {r["period_key"] for r in control.select("period_key").collect()}
latest = max(existing)
latest_year, latest_month = int(latest[:4]), int(latest[5:7])

today = datetime.now(timezone.utc)
candidates = []
y, mth = latest_year, latest_month
for _ in range(MAX_MONTHS_TO_CHECK):
    mth += 1
    if mth > 12:
        y, mth = y + 1, 1
    if (y, mth) > (today.year, today.month):
        break
    candidates.append((y, mth))

print(f"Latest month in control table: {latest}")
print(f"Candidates to check: {[f'{y}-{m:02d}' for y, m in candidates] or 'none'}")


def file_exists(url: str) -> bool:
    try:
        r = requests.get(url, headers={"Range": "bytes=0-0"}, timeout=(30, 120), stream=True)
        r.close()
        return r.status_code in (200, 206)
    except Exception as e:
        print(f"  Check failed for {url}: {str(e)[:150]}")
        return False


new_rows = []
for y, mth in candidates:
    file_name = f"{FILE_PREFIX}{y}_{mth}.zip"
    url = f"{BASE_URL}{file_name}"
    if file_exists(url):
        print(f"  {y}-{mth:02d}: available")
        new_rows.append((y, mth, f"{y}-{mth:02d}", file_name, url, "pending", "pending"))
    else:
        print(f"  {y}-{mth:02d}: not published yet. Stopping.")
        break

if new_rows:
    (
        spark.createDataFrame(
            new_rows,
            ["year", "month", "period_key", "file_name", "source_url", "download_status", "load_status"],
        )
        .withColumn("last_updated", F.current_timestamp())
        .write.format("delta").mode("append").saveAsTable(CONTROL_TABLE)
    )

print(f"\nMonths added to control table: {len(new_rows)} {[r[2] for r in new_rows]}")
print(f"Control table now holds {spark.table(CONTROL_TABLE).count()} months")

Latest month in control table: 2026-06
Candidates to check: ['2026-07', '2026-08', '2026-09']
  2026-07: not published yet. Stopping.

Months added to control table: 0 []
Control table now holds 90 months
